In [ ]:
import importlib
import setup_biashara

importlib.reload(setup_biashara)

setup_biashara.create_biashara_database()

In [ ]:
!python -m streamlit run app.py

In [5]:
import importlib
import sqlite3
import setup_biashara
import operations

importlib.reload(setup_biashara)
importlib.reload(operations)

setup_biashara.create_biashara_database()

conn = sqlite3.connect(setup_biashara.DB_NAME)
views = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='view'"
).fetchall()
conn.close()

print("Views in database:", [v[0] for v in views])

Views in database: ['daily_business_summary', 'daily_sales_view']


In [14]:
import json
import subprocess
from pathlib import Path
from PIL import Image

base_dir = Path(".")
static_dir = base_dir / "static"
static_dir.mkdir(exist_ok=True)

svg_file = base_dir / "logo.svg"
logo_png = base_dir / "logo.png"

if not svg_file.exists():
    raise FileNotFoundError(f"Missing {svg_file.resolve()}")

rendered = False

# 1. Best fidelity: cairosvg if installed
try:
    import cairosvg

    cairosvg.svg2png(
        url=str(svg_file.resolve()),
        write_to=str(logo_png.resolve()),
        output_width=512,
        output_height=512,
    )
    rendered = True
    print("Rendered logo.png via cairosvg")
except Exception:
    pass

# 2. High fidelity: headless Edge or Chrome screenshot of logo.svg
if not rendered:
    browser_paths = [
        r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
        r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
    ]
    for exe in browser_paths:
        if Path(exe).exists():
            try:
                subprocess.run(
                    [
                        exe,
                        "--headless",
                        "--disable-gpu",
                        "--hide-scrollbars",
                        "--window-size=512,512",
                        f"--screenshot={logo_png.resolve()}",
                        svg_file.resolve().as_uri(),
                    ],
                    check=True,
                    timeout=20,
                )
                if logo_png.exists():
                    img = Image.open(logo_png).convert("RGBA")
                    img = img.crop((0, 0, min(512, img.width), min(512, img.height)))
                    # Make near-white outer corners transparent so there's no white box border
                    px = img.load()
                    w, h = img.size
                    for y in range(h):
                        for x in range(w):
                            r, g, b, a = px[x, y]
                            if r > 245 and g > 245 and b > 245:
                                px[x, y] = (r, g, b, 0)
                    img.save(logo_png)
                    rendered = True
                    print(f"Rendered logo.png via {Path(exe).name} with transparent outer corners")
                    break
            except Exception as err:
                print(f"Browser render failed ({Path(exe).name}): {err}")

# 3. Fallback: inspect SVG text if neither cairosvg nor a browser screenshot worked
if not rendered:
    print("Could not render SVG automatically. Here is logo.svg so we can match every coordinate:")
    print(svg_file.read_text(encoding="utf-8", errors="replace"))
else:
    base_img = Image.open(logo_png).convert("RGBA")
    base_img.resize((192, 192), Image.Resampling.LANCZOS).save(static_dir / "icon-192.png")
    base_img.resize((512, 512), Image.Resampling.LANCZOS).save(static_dir / "icon-512.png")

    # Write static/manifest.json
    manifest = {
        "name": "Biashara Business Manager",
        "short_name": "Biashara",
        "start_url": "/",
        "display": "standalone",
        "background_color": "#ffffff",
        "theme_color": "#16764a",
        "icons": [
            {
                "src": "/app/static/icon-192.png",
                "sizes": "192x192",
                "type": "image/png",
            },
            {
                "src": "/app/static/icon-512.png",
                "sizes": "512x512",
                "type": "image/png",
            },
        ],
    }
    manifest_path = static_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    # Ensure static file serving is enabled in .streamlit/config.toml
    config_dir = base_dir / ".streamlit"
    config_dir.mkdir(exist_ok=True)
    config_path = config_dir / "config.toml"
    config_path.write_text('[server]\nenableStaticServing = true\n', encoding="utf-8")

    print("Saved:")
    print(" -", logo_png.resolve())
    print(" -", (static_dir / "icon-192.png").resolve())
    print(" -", (static_dir / "icon-512.png").resolve())
    print(" -", manifest_path.resolve())
    print(" -", config_path.resolve())

Rendered logo.png via chrome.exe with transparent outer corners
Saved:
 - C:\Users\HomePC\Desktop\Mariita Analytics Projects\biashara app\logo.png
 - C:\Users\HomePC\Desktop\Mariita Analytics Projects\biashara app\static\icon-192.png
 - C:\Users\HomePC\Desktop\Mariita Analytics Projects\biashara app\static\icon-512.png
 - C:\Users\HomePC\Desktop\Mariita Analytics Projects\biashara app\static\manifest.json
 - C:\Users\HomePC\Desktop\Mariita Analytics Projects\biashara app\.streamlit\config.toml
